# Connect Four with Fixed-Depth Minimax

**AIMLZG557 Artificial and Computational Intelligence, S1-26**

Assignment 2 (Situated Learning), Question 1 | **Group 107**

A human plays against the computer on a 7x6 grid. The computer chooses its moves with Minimax at a fixed depth of 3, using a static evaluation function designed by hand. No machine learning is used, and the code needs only the Python standard library.

**How to run:** choose *Run All*. Every cell runs by itself until the last one, which waits for you to type your moves.

**Where each requirement is met**

| Requirement in the brief | Where |
|---|---|
| (a) Own static evaluation function, justified and explained with a sample state, no machine learning | Part (a), code in c.3 |
| (b) Human gives moves during play, computer plays by fixed-depth Minimax, turns alternate | Part (b), c.4, c.5, *Play the game* |
| (c) Python implementation using Minimax | Part (c) |
| 7x6 grid as a 2D array | c.1 |
| Human picks a column, full columns are refused | c.5, Verification |
| Minimax at depth 3 that considers winning, blocking and strategic positioning | c.3, c.4, Verification |
| Alternate turns, update the grid, check for a win or a draw after each move | c.2, c.5 |
| Show the grid after each move, announce the winner or a draw | c.5, Demonstration games |


## Part (a): Static evaluation function

The search stops after three moves, usually before the game is over, so it needs a score for unfinished positions. `evaluate(board)` gives that score from the computer's side: positive favours the computer and negative favours the human.

### The design

There are 69 possible lines of four on the grid. If a line holds discs from both players, nobody can ever finish it, so it is **dead** and scores 0. Otherwise it is **live** for the player whose discs are on it, and it scores by how many discs it holds.

| What is counted | Score | Factor from the brief |
|---|---|---|
| Live line with 1 disc | +1 for the computer, -1 for the human | Strategic positioning |
| Live line with 2 discs | +3 for the computer, -3 for the human | Winning and blocking |
| Live line with 3 discs | +9 for the computer, -9 for the human | Winning and blocking |
| Human three whose empty cell can be played right now | a further -100 | Blocking |
| Four in a row (a terminal state, handled by Minimax) | +/-(1,000,000 + remaining depth) | Winning |

### Why these values

**Positioning comes from the line counts.** A disc scores once for every live line through its cell. A centre cell lies on 13 lines and a corner on only 3 (the full table is printed in c.3), so central play is rewarded without a separate centre bonus, and that reward falls when the opponent kills those lines.

**Each extra disc triples a line's value.** Every disc on a line leaves one move fewer to finish it, so the value should rise quickly with the disc count.

**Blocking is built in.** The human's lines use the same values with the opposite sign, so a move that kills a human line raises the score.

**The -100 term matters most.** At depth 3 the search runs computer, human, computer, then evaluates. So every unfinished position that reaches `evaluate` has the **human to move**. If a human three has an empty cell that can be played now, the human wins on the very next move, one move beyond what the search can see. Only the evaluation can warn about this.
- It applies only when the empty cell is playable. A gap above an empty cell cannot be filled yet.
- It applies only to the human. A computer three in the same position can simply be blocked, because the human moves next. Testing confirmed this: giving the computer the same style of bonus made it weaker.

**Why 100?** It has to outrank ordinary line totals without looking like a real loss. The largest plain line total we saw was 91, across 669,636 evaluated positions, so 100 clears it while staying 10,000 times smaller than the win score. A penalty of 1,000 scored exactly level with 100, so nothing is gained by going higher.

### Testing the design

We tested each choice instead of assuming it. Each row plays the final design against one altered design, both at depth 3, for 1,000 games from random four-move openings, with each opening played from both sides. A score above 50% means the final design is stronger, and about +/-3% is noise at this sample size. These runs take several minutes, so they were done with a separate script and are not repeated in this notebook.

| Altered design | Final design's score | What it shows |
|---|---|---|
| No evaluation (every leaf scores 0) | 79.0% | The evaluation adds a lot of strength |
| No next-move threat penalty | 66.1% | The -100 term is worth having |
| No single-disc lines (0/3/9) | 62.0% | The positioning term matters |
| Much larger value for threes (1/3/27) | 60.8% | Threes should not dominate |
| Same style of bonus for the computer's own threes (+30) | 55.5% | The threat term belongs to the human only |
| Steeper scale (1/4/16) | 53.6% | Slightly weaker than 1/3/9 |
| Flatter scale (1/2/4) | 51.5% | No clear difference |
| Extra centre-column bonus (+2 per disc) | 51.3% | No clear gain, so we left it out |
| Threat penalty of 1,000 instead of 100 | 50.0% | Exactly level, so 100 is enough |

### Sample game state

`X` is the human and `O` is the computer. Rows are numbered from the bottom, and it is the human's turn.

```
1 2 3 4 5 6 7
. . . . . . .   row 6
. . . . . . .   row 5
. . . O . . .   row 4
. . . O . . .   row 3
. . X O . . .   row 2
. . X X X O .   row 1
```

This is a position the search really evaluates. Take away the disc in column 6, the disc in column 5 and the top disc in column 4, and it is the computer's move. The search then reaches this state through computer column 6, human column 5, computer column 4.

| Feature | Count | Value | Subtotal |
|---|---:|---:|---:|
| Computer live lines with 1 disc | 21 | +1 | +21 |
| Computer live lines with 2 discs | 2 | +3 | +6 |
| Computer live lines with 3 discs | 1 | +9 | +9 |
| Human live lines with 1 disc | 3 | -1 | -3 |
| Human live lines with 2 discs | 3 | -3 | -9 |
| Human live lines with 3 discs | 1 | -9 | -9 |
| Human threes playable next move | 1 | -100 | -100 |
| **Total** | | | **-85** |

The lines with two or more discs are:
- **Computer three:** column 4, rows 2 to 5, with row 5 empty.
- **Computer twos:** column 4, rows 3 to 6, and the diagonal from column 3 row 4 down to column 6 row 1.
- **Human three:** row 1, columns 2 to 5, with column 2 empty. That cell is on the bottom row, so it can be played now.
- **Human twos:** column 3, rows 1 to 4; row 1, columns 1 to 4; and the diagonal from column 1 row 4 down to column 4 row 1.

Besides these seven lines, 24 lines hold a single disc, 10 are dead (for example row 1, columns 3 to 6, which holds `X X X O`) and 28 are empty. That accounts for all 69.

**What the number tells us.** The line counts alone give 36 - 21 = +15, so the position looks good for the computer, which has its own three and more live lines. But it is the human's move, and column 2 wins at once. The -100 term corrects this to **-85**, and the search steers away from this position.

If the computer had blocked in column 2 instead of stacking in column 4, the score would be **+10**. The search therefore prefers the block even though it gives up the computer's own three. The code in c.3 confirms both values.


## Part (b): Game design

| Element | Design |
|---|---|
| State | 2D array `board[row][column]` with 6 rows and 7 columns. Each cell is `0` empty, `1` human (X) or `2` computer (O) |
| Initial state | All cells empty, and the human chooses who starts |
| Actions | Any column whose top cell is empty |
| Transition | The disc lands on the lowest empty cell of the chosen column |
| Terminal test | Four in a row for either player, or no empty column left |
| Utility | +(1,000,000 + remaining depth) for a computer win, the negative of that for a human win, 0 for a draw |

**Turn flow**
1. Ask who starts and show the empty grid.
2. On the human's turn, read a column from the keyboard. Non-numbers, columns that do not exist and full columns are refused, and the question is asked again.
3. On the computer's turn, run Minimax, drop the disc and print the value found for each column.
4. After every move, show the grid and check for a win, then for a draw. If the game continues, the other player moves.

Columns are numbered 1 to 7 from the left, matching the sample moves in the brief. The code uses 0 to 6 internally.

**Minimax.** The depth is fixed at 3 and counts single moves: computer, human, computer, then evaluate. From an empty grid that is 7 + 49 + 343 = 399 positions, so a move takes a fraction of a second and no pruning is needed. Win, loss and draw are checked at every node, not only at the leaves. Adding the remaining depth to the win score makes a sooner win worth more, so the computer does not delay a win, and it holds off a loss for as long as it can.

**How the three factors in the brief are handled**
- **Winning:** a move that completes four gets the highest value.
- **Blocking:** if the human could win next move, every move that fails to stop it scores about -1,000,000, so Minimax picks the block. The -100 term covers the same danger one move past the search limit.
- **Strategic positioning:** in quiet positions, the evaluation's line counts steer the computer towards cells that lie on many live lines.


## Part (c): Implementation

### c.1 The board


In [1]:
ROWS = 6
COLUMNS = 7

EMPTY = 0
HUMAN = 1
COMPUTER = 2
SYMBOLS = {EMPTY: ".", HUMAN: "X", COMPUTER: "O"}


def create_board():
    """Return an empty grid as a 2D array: board[row][column], row 0 is the top."""
    return [[EMPTY for _ in range(COLUMNS)] for _ in range(ROWS)]


def valid_columns(board):
    """A column can be played while its top cell is empty."""
    return [column for column in range(COLUMNS) if board[0][column] == EMPTY]


def drop_disc(board, column, player):
    """Let a disc fall to the lowest empty cell of the column and return its row."""
    for row in range(ROWS - 1, -1, -1):
        if board[row][column] == EMPTY:
            board[row][column] = player
            return row
    raise ValueError("Column {} is full".format(column + 1))


def print_board(board):
    """Show the grid with the 1-7 column numbers the human player types."""
    print(" ".join(str(column + 1) for column in range(COLUMNS)))
    for row in board:
        print(" ".join(SYMBOLS[cell] for cell in row))
    print()

### c.2 Win and draw checks

`ALL_LINES` holds the 69 lines of four once, so the win check and the evaluation both work from the same list.


In [2]:
def build_lines():
    """List every group of four cells in a straight line (69 on a 7x6 grid)."""
    directions = [(0, 1), (1, 0), (1, 1), (-1, 1)]  # right, down, down-right, up-right
    lines = []
    for row in range(ROWS):
        for column in range(COLUMNS):
            for row_step, column_step in directions:
                cells = [(row + i * row_step, column + i * column_step) for i in range(4)]
                if all(0 <= r < ROWS and 0 <= c < COLUMNS for r, c in cells):
                    lines.append(cells)
    return lines


ALL_LINES = build_lines()


def has_won(board, player):
    """True if the player owns all four cells of any line."""
    return any(all(board[r][c] == player for r, c in line) for line in ALL_LINES)


def is_draw(board):
    """A draw is a full grid with no winner."""
    return not valid_columns(board) and not has_won(board, HUMAN) and not has_won(board, COMPUTER)

### c.3 Static evaluation

This is the design from Part (a). `count_features` counts what the table describes and `evaluate` applies the values.


In [3]:
LINE_VALUE = {1: 1, 2: 3, 3: 9}
NEXT_MOVE_THREAT = 100


def is_playable(board, row, column):
    """An empty cell can be filled now if it is on the bottom row or rests on a disc."""
    return row == ROWS - 1 or board[row + 1][column] != EMPTY


def count_features(board):
    """Count the live lines of each kind, plus the human threes that can be finished next move."""
    features = {
        "computer": {1: 0, 2: 0, 3: 0},
        "human": {1: 0, 2: 0, 3: 0},
        "human_next_move_threats": 0,
    }
    for line in ALL_LINES:
        cells = [board[r][c] for r, c in line]
        computer_discs = cells.count(COMPUTER)
        human_discs = cells.count(HUMAN)
        if computer_discs and human_discs:
            continue  # dead line: neither player can ever complete it
        if computer_discs:
            features["computer"][computer_discs] += 1
        elif human_discs:
            features["human"][human_discs] += 1
            if human_discs == 3:
                row, column = line[cells.index(EMPTY)]
                if is_playable(board, row, column):
                    features["human_next_move_threats"] += 1
    return features


def evaluate(board):
    """Static value of an unfinished position from the computer's side (positive is good for it)."""
    features = count_features(board)
    score = -NEXT_MOVE_THREAT * features["human_next_move_threats"]
    for discs, value in LINE_VALUE.items():
        score += value * (features["computer"][discs] - features["human"][discs])
    return score

The cell below rebuilds the sample state from Part (a), confirms that `evaluate` gives the hand tally, and prints the line count of each cell.


In [4]:
def board_from(text):
    """Build a 2D array from a drawing with . X O, top row first."""
    symbol_to_cell = {symbol: cell for cell, symbol in SYMBOLS.items()}
    return [[symbol_to_cell[symbol] for symbol in row.split()] for row in text.strip().splitlines()]


SAMPLE_STATE = board_from("""
. . . . . . .
. . . . . . .
. . . O . . .
. . . O . . .
. . X O . . .
. . X X X O .
""")

COMPUTER_BLOCKS_INSTEAD = board_from("""
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . X O . . .
. O X X X O .
""")

print("Sample state (human to move):")
print_board(SAMPLE_STATE)
print("Features:", count_features(SAMPLE_STATE))

hand_tally = (21 * 1 + 2 * 3 + 1 * 9) - (3 * 1 + 3 * 3 + 1 * 9) - 1 * 100
print("Hand tally from Part (a):", hand_tally)
print("evaluate(SAMPLE_STATE):  ", evaluate(SAMPLE_STATE))
assert evaluate(SAMPLE_STATE) == hand_tally == -85

print("\nIf the computer had blocked in column 2 instead:")
print_board(COMPUTER_BLOCKS_INSTEAD)
print("evaluate(COMPUTER_BLOCKS_INSTEAD):", evaluate(COMPUTER_BLOCKS_INSTEAD))
assert evaluate(COMPUTER_BLOCKS_INSTEAD) == 10

print("\nNumber of lines through each cell, the source of the positioning term:")
for row in range(ROWS):
    print(" ".join("{:2d}".format(sum((row, column) in line for line in ALL_LINES)) for column in range(COLUMNS)))

Sample state (human to move):
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . O . . .
. . . O . . .
. . X O . . .
. . X X X O .

Features: {'computer': {1: 21, 2: 2, 3: 1}, 'human': {1: 3, 2: 3, 3: 1}, 'human_next_move_threats': 1}
Hand tally from Part (a): -85
evaluate(SAMPLE_STATE):   -85

If the computer had blocked in column 2 instead:
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . X O . . .
. O X X X O .

evaluate(COMPUTER_BLOCKS_INSTEAD): 10

Number of lines through each cell, the source of the positioning term:
 3  4  5  7  5  4  3
 4  6  8 10  8  6  4
 5  8 11 13 11  8  5
 5  8 11 13 11  8  5
 4  6  8 10  8  6  4
 3  4  5  7  5  4  3


### c.4 Minimax at fixed depth 3

The computer maximises the value and the human is assumed to minimise it. `choose_computer_column` makes the computer's first move, so `minimax` starts with `SEARCH_DEPTH - 1` plies left. Trial moves are undone after each call, so the real board is never changed by the search.


In [5]:
SEARCH_DEPTH = 3
WIN_SCORE = 1000000
CENTRE_FIRST = [3, 2, 4, 1, 5, 0, 6]  # ties go to the column nearest the centre


def ordered_moves(board):
    playable = valid_columns(board)
    return [column for column in CENTRE_FIRST if column in playable]


def minimax(board, depth, computer_to_move):
    """Return the value of this position for the computer, looking `depth` more plies ahead."""
    if has_won(board, COMPUTER):
        return WIN_SCORE + depth  # a larger remaining depth means the win comes sooner
    if has_won(board, HUMAN):
        return -(WIN_SCORE + depth)  # a loss that comes sooner is worse
    if not valid_columns(board):
        return 0
    if depth == 0:
        return evaluate(board)

    player = COMPUTER if computer_to_move else HUMAN
    values = []
    for column in ordered_moves(board):
        row = drop_disc(board, column, player)
        values.append(minimax(board, depth - 1, not computer_to_move))
        board[row][column] = EMPTY  # undo the trial move
    return max(values) if computer_to_move else min(values)


def choose_computer_column(board):
    """Try each column at the root and return (best_column, {column: value})."""
    values = {}
    for column in ordered_moves(board):
        row = drop_disc(board, column, COMPUTER)
        values[column] = minimax(board, SEARCH_DEPTH - 1, False)
        board[row][column] = EMPTY
    best_column = max(values, key=values.get)  # first maximum, so ties favour the centre
    return best_column, values

### c.5 Human input and the game loop

The human types a column during play. `play_game` alternates turns, shows the grid after each move and announces the result.


In [6]:
def ask_first_player(read_line):
    """Ask who starts. Returns True/False, or None if no keyboard input is available."""
    while True:
        try:
            answer = read_line("Do you want to play first? (y/n): ").strip().lower()
        except (EOFError, NotImplementedError):
            return None
        if answer in ("y", "yes"):
            return True
        if answer in ("n", "no"):
            return False
        print("Please type y or n.")


def ask_human_column(board, read_line):
    """Keep asking until the human gives a column that exists and is not full."""
    while True:
        try:
            text = read_line("Your move, choose a column (1-{}): ".format(COLUMNS)).strip()
        except (EOFError, NotImplementedError):
            return None
        try:
            column = int(text) - 1
        except ValueError:
            print("'{}' is not a number. Please type a column from 1 to {}.".format(text, COLUMNS))
            continue
        if not 0 <= column < COLUMNS:
            print("Column {} does not exist. Please type a column from 1 to {}.".format(text, COLUMNS))
        elif column not in valid_columns(board):
            print("Column {} is full. Please choose another column.".format(column + 1))
        else:
            return column


def play_game(read_line=None, board=None):
    """Play one game and return 'human', 'computer', 'draw' or 'stopped'.

    read_line defaults to input() for real play. Tests pass a scripted reader instead.
    board lets a test start from a prepared position.
    """
    read_line = input if read_line is None else read_line
    board = create_board() if board is None else board
    human_first = ask_first_player(read_line)
    if human_first is None:
        print("No keyboard input is available. Run this cell in Jupyter to play.")
        return "stopped"

    print("\nYou are X, the computer is O. Starting grid:")
    print_board(board)
    player = HUMAN if human_first else COMPUTER

    while True:
        if player == HUMAN:
            column = ask_human_column(board, read_line)
            if column is None:
                print("No keyboard input is available, so the game stops here.")
                return "stopped"
            drop_disc(board, column, HUMAN)
            print("You dropped a disc into column {}.".format(column + 1))
        else:
            column, values = choose_computer_column(board)
            drop_disc(board, column, COMPUTER)
            ranked = ", ".join("{}: {}".format(c + 1, values[c]) for c in sorted(values))
            print("Minimax values by column -> {}".format(ranked))
            print("Computer dropped a disc into column {}.".format(column + 1))
        print_board(board)

        if has_won(board, player):
            print("You win!" if player == HUMAN else "The computer wins!")
            return "human" if player == HUMAN else "computer"
        if is_draw(board):
            print("The grid is full. The game is a draw.")
            return "draw"
        player = COMPUTER if player == HUMAN else HUMAN

## Verification

These checks run the code above against each requirement. Any failure stops the notebook with an error, so reaching the end of this section means every check passed.


In [7]:
import contextlib
import io
import random


def check(description, passed):
    assert passed, "FAILED: " + description
    print("PASS -", description)


def quietly(function, *args, **kwargs):
    """Run a function, hide what it prints, and return (result, printed_text)."""
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        result = function(*args, **kwargs)
    return result, captured.getvalue()


def scripted_reader(answers):
    """Stand-in for input(): replays prepared answers and echoes them like a real session."""
    remaining = iter(answers)

    def read_line(prompt):
        answer = next(remaining)
        print(prompt + answer)
        return answer

    return read_line

**Board, input and rules**


In [8]:
grid = create_board()
check("The grid is a 2D array of 6 rows and 7 columns",
      len(grid) == 6 and all(len(row) == 7 for row in grid))
check("There are 69 possible lines of four", len(ALL_LINES) == 69)

drop_disc(grid, 2, HUMAN)
drop_disc(grid, 3, COMPUTER)
drop_disc(grid, 2, HUMAN)
check("Sample moves from the brief: the second disc in column 3 stacks on the first",
      grid[5][2] == HUMAN and grid[4][2] == HUMAN and grid[5][3] == COMPUTER)

grid = create_board()
for _ in range(ROWS):
    drop_disc(grid, 2, HUMAN)
check("A full column is no longer a valid move", 2 not in valid_columns(grid))

column, log = quietly(ask_human_column, grid, scripted_reader(["abc", "0", "8", "3", "5"]))
check("Text, out-of-range numbers and a full column are rejected, then column 5 is accepted",
      column == 4 and "not a number" in log and "does not exist" in log and "is full" in log)

winning_positions = {
    "horizontal": """
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X X X X . .
""",
    "vertical": """
. . . . . . .
. . . . . . .
. . . X . . .
. . . X . . .
. . . X . . .
. . . X . . .
""",
    "rising diagonal": """
. . . . . . .
. . . . . . .
. . . . X . .
. . . X O . .
. . X O O . .
. X O O O . .
""",
    "falling diagonal": """
. . . . . . .
. . . . . . .
. . X . . . .
. . O X . . .
. . O O X . .
. . O O O X .
""",
}
for name, drawing in winning_positions.items():
    position = board_from(drawing)
    check("A {} four in a row is a win".format(name),
          has_won(position, HUMAN) and not has_won(position, COMPUTER))

position = board_from("""
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
X . . . . . .
. . . . X X X
""")
check("Three in a row is not a win, and lines do not wrap round the edge", not has_won(position, HUMAN))

DRAWN_GRID = """
X X O O X X O
O O X X O O X
X X O O X X O
O O X X O O X
X X O O X X O
O O X X O O X
"""
check("A full grid with no four in a row is a draw", is_draw(board_from(DRAWN_GRID)))
check("A won position is not a draw", not is_draw(board_from(winning_positions["vertical"])))

PASS - The grid is a 2D array of 6 rows and 7 columns
PASS - There are 69 possible lines of four
PASS - Sample moves from the brief: the second disc in column 3 stacks on the first
PASS - A full column is no longer a valid move
PASS - Text, out-of-range numbers and a full column are rejected, then column 5 is accepted
PASS - A horizontal four in a row is a win
PASS - A vertical four in a row is a win
PASS - A rising diagonal four in a row is a win
PASS - A falling diagonal four in a row is a win
PASS - Three in a row is not a win, and lines do not wrap round the edge
PASS - A full grid with no four in a row is a draw
PASS - A won position is not a draw


**Computer decisions**


In [9]:
position = board_from("""
. . . . . . .
. . . . . . .
. . . . . . .
. . O . . X .
. . O . . X .
. . O . . X .
""")
column, values = choose_computer_column(position)
check("Winning: the computer completes its own four rather than blocking the human's",
      column == 2 and values[2] == WIN_SCORE + 2)

position = board_from("""
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . O O . . .
O X X X . . .
""")
column, values = choose_computer_column(position)
check("Blocking: the computer plays column 5 to stop the human's four", column == 4)

column, values = choose_computer_column(create_board())
check("Positioning: on an empty grid the computer chooses the centre column", column == 3)
check("Positioning: the centre scores higher than any other opening, not just on a tie",
      all(values[3] > value for other, value in values.items() if other != 3))


def positions_evaluated_on_empty_grid():
    global evaluate
    real_evaluate = evaluate
    calls = []

    def counting_evaluate(board):
        calls.append(1)
        return real_evaluate(board)

    evaluate = counting_evaluate
    try:
        choose_computer_column(create_board())
    finally:
        evaluate = real_evaluate
    return len(calls)


check("Depth is exactly 3: 7 x 7 x 7 = 343 positions are evaluated from an empty grid",
      SEARCH_DEPTH == 3 and positions_evaluated_on_empty_grid() == 343)

PASS - Winning: the computer completes its own four rather than blocking the human's
PASS - Blocking: the computer plays column 5 to stop the human's four
PASS - Positioning: on an empty grid the computer chooses the centre column
PASS - Positioning: the centre scores higher than any other opening, not just on a tie
PASS - Depth is exactly 3: 7 x 7 x 7 = 343 positions are evaluated from an empty grid


**Game flow**


In [10]:
near_full = board_from(DRAWN_GRID)
near_full[0][0] = EMPTY
result, log = quietly(play_game, scripted_reader(["y", "1"]), near_full)
check("The game announces a draw when the last cell is filled with no winner",
      result == "draw" and "draw" in log)

result, log = quietly(play_game, scripted_reader(["n"] + [str(i % 7 + 1) for i in range(100)]))
first_move = next(line for line in log.splitlines() if "dropped a disc" in line)
check("If the human chooses not to start, the computer moves first",
      first_move == "Computer dropped a disc into column 4.")


def no_keyboard(prompt):
    raise EOFError


result, log = quietly(play_game, no_keyboard)
check("With no keyboard attached the game stops cleanly instead of crashing", result == "stopped")


def match_against_random_player(games, seed):
    rng = random.Random(seed)
    results = {"computer": 0, "random player": 0, "draw": 0}
    for game in range(games):
        board = create_board()
        player = COMPUTER if game % 2 == 0 else HUMAN
        while True:
            if player == COMPUTER:
                column, _ = choose_computer_column(board)
            else:
                column = rng.choice(valid_columns(board))
            drop_disc(board, column, player)
            if has_won(board, player):
                results["computer" if player == COMPUTER else "random player"] += 1
                break
            if is_draw(board):
                results["draw"] += 1
                break
            player = COMPUTER if player == HUMAN else HUMAN
    return results


results = match_against_random_player(games=50, seed=557)
print("\nDepth-3 computer against a random player over 50 games:", results)
check("The computer wins the clear majority against a random player", results["computer"] >= 45)

PASS - The game announces a draw when the last cell is filled with no winner
PASS - If the human chooses not to start, the computer moves first
PASS - With no keyboard attached the game stops cleanly instead of crashing



Depth-3 computer against a random player over 50 games: {'computer': 50, 'random player': 0, 'draw': 0}
PASS - The computer wins the clear majority against a random player


## Demonstration games (scripted)

Nobody is typing in these two games. A script feeds the human's moves through the same `play_game` function used in real play, so that each ending and the input checks are visible here. The real interactive game is in the last cell.

**Game 1: the computer wins.** The script also types two invalid entries to show that they are refused.


In [11]:
result = play_game(scripted_reader(["y", "abc", "9", "2", "4", "6", "4", "7", "3", "5", "2", "5"]))
assert result == "computer"

Do you want to play first? (y/n): y

You are X, the computer is O. Starting grid:
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .

Your move, choose a column (1-7): abc
'abc' is not a number. Please type a column from 1 to 7.
Your move, choose a column (1-7): 9
Column 9 does not exist. Please type a column from 1 to 7.
Your move, choose a column (1-7): 2
You dropped a disc into column 2.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X . . . . .

Minimax values by column -> 1: 1, 2: 2, 3: 2, 4: 5, 5: 3, 6: 1, 7: 0
Computer dropped a disc into column 4.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X . O . . .

Your move, choose a column (1-7): 4
You dropped a disc into column 4.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. X . O . . .

Minimax values by column -> 1: -4, 2: 1, 3: -3, 4: 2, 5: -1, 6: -1, 7: -2
Computer d

**Game 2: the human wins.** The human builds a double threat, with winning moves in both column 3 and column 7. Depth 3 sees this too late, and the computer cannot block both.


In [12]:
result = play_game(scripted_reader(["y", "2", "6", "5", "3", "5", "3", "4", "5", "6", "3"]))
assert result == "human"

Do you want to play first? (y/n): y

You are X, the computer is O. Starting grid:
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .

Your move, choose a column (1-7): 2
You dropped a disc into column 2.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X . . . . .

Minimax values by column -> 1: 1, 2: 2, 3: 2, 4: 5, 5: 3, 6: 1, 7: 0
Computer dropped a disc into column 4.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X . O . . .

Your move, choose a column (1-7): 6
You dropped a disc into column 6.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. X . O . X .

Minimax values by column -> 1: 3, 2: 7, 3: 3, 4: 4, 5: 3, 6: 7, 7: 3
Computer dropped a disc into column 2.
1 2 3 4 5 6 7
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O . . . . .
. X . O . X .

Your move, choose a column (1-7): 5
You dropped a disc into column 

## Play the game

Run the cell below. You will be asked whether you want to go first, then for a column number from 1 to 7 on each turn. You are `X` and the computer is `O`.


In [ ]:
play_game()